# pre-kamp2 진행 상황 정리 (시간순 기록)

이 문서는 2026 K-인공지능 제조데이터 경진대회(6회) 연습 프로젝트 **pre-kamp2**의 작업 경과를
시간순으로 정리한 기록입니다. 노트북/코드의 상세 내용은 각 노트북과 `README.md`,
`reports/01_model_comparison_report.md`를 참고하세요.

작성 기준일: **2026-09-16**

---

## 1단계 (2026-09-15) — 프로젝트 현황 파악 및 01 베이스라인 검증

- 로컬 PC(`hy`)에 연결하여 `pre-kamp2` 폴더 구조를 확인.
- 사용자가 `01_baseline_linear_regression.ipynb`를 로컬에서 전부 실행 완료했다고 알려와,
  실행 결과(셀 1~23, 오류 없음)를 검증.
  - 원본 센서 CSV(약 294만 행) + 라벨(136건)을 배정번호 단위로 min/max 집계(38개 피처) 후
    병합, 다중공선성 제거(27개 피처), Train:Val:Test = 108:14:14 분할.
  - LinearRegression과 SGDRegressor(체크포인트 재개 포함)를 비교해 **SGDRegressor를
    베이스라인으로 선정** — Test R²=0.0414, RMSE=0.0532, MAE=0.0334.

---

## 2단계 (2026-09-15) — Unnamed 컬럼 문제 발견 및 02 노트북 개발

- 사용자가 "unnamed관련된 컬럼은 어떤거야? 처리해야할거같아"라고 질문.
  → 원본 `품질전처리후데이터.csv`를 저장할 때 pandas 인덱스가 그대로 컬럼(`Unnamed: 0`)으로
    딸려 들어갔고, 이것이 01의 최종 모델 입력 변수(`Unnamed: 0_max`)에 실수로 포함되어 있던
    문제를 확인.
- 사용자 요청에 따라 `02_preprocessing_and_skew_correction.ipynb` 신규 제작:
  1. 원본 행 번호 정보를 `original_row_range.csv`로 별도 저장 후, 분석 데이터에서는
     Unnamed 컬럼 완전 제거.
  2. 독립변수 간 상관관계 히트맵(다중공선성 제거 전/후) 작성.
  3. 왜도(skew) 진단 후 Yeo-Johnson 파워변환(`PowerTransformer`) 적용 — train에만 fit,
     val/test는 transform만 적용해 데이터 누수 방지.
- **버그 수정**: PowerTransformer를 여러 컬럼에 한 번에 `fit()`하면 일부 컬럼에서
  `BracketError`(scipy 최적화 실패)가 발생해 전체가 실패하는 문제 발견 → 컬럼별로 개별
  `try/except`로 감싸 격리하도록 `training_utils.py`의 `correct_skew_with_power_transform()`을
  재작성. 재현 테스트로 검증.
- 사용자 요청으로 변환 전/후 분포 그래프를 "데이터가 중앙으로 이동하는 느낌"이 들도록
  개선(중심선 표시, 대칭 x축 범위).
- 사용자가 02 노트북을 로컬에서 전부 실행 완료(오류 없음) 확인.

---

## 3단계 (2026-09-15~16) — 03 노트북 계획 수립 및 개발

사용자가 다음 4가지 작업을 요청:

1. 전처리한 데이터로 이전 선형회귀 모델 성능 재평가
2. 트리 계열 모델 대표적인 것 몇 개 성능 평가
3. 릿지/라쏘 정규화, 트리 모델도 설명 후 괜찮으면 진행
4. 모든 모델 평가 후 전체 평가 및 1차 보고서 작성

확인 질문(AskUserQuestion)에 대한 사용자 답변:

- 트리 모델 범위: **DecisionTree + RandomForest + GradientBoosting**
- 1차 보고서 형식: **`.md`와 `.pdf` 별도 생성**

`03_model_comparison.ipynb` 개발 내용:

- 02의 전처리 데이터에 `MinMaxScaler`(train에만 fit)를 추가 적용.
- LinearRegression, SGDRegressor, Ridge, Lasso, DecisionTree, RandomForest, GradientBoosting
  7개 모델 학습 및 비교.
- Ridge/Lasso/DecisionTree는 `grid_search_by_val()`(신규 함수, Validation 기반 그리드서치)로
  하이퍼파라미터 탐색.
- RandomForest는 `train_random_forest_with_checkpoint()`(신규 함수, `warm_start` 기반
  체크포인트 학습)로 학습.
- GradientBoosting은 `staged_predict()`로 부스팅 반복별 학습곡선을 재구성해 최적
  반복 수를 사후 선택.
- 합성 데이터로 전체 셀을 순차 실행하는 **드라이런(dry-run) 스크립트**로 사전 검증 완료.

작업 중 발생한 이슈:

- 사용자가 `BracketError` 트래백을 재차 보고 → 파일을 다시 스테이징해 line 694를 직접
  확인한 결과 이미 수정된 코드였음을 확인. **Jupyter 커널이 이전에 import한 모듈을
  메모리에 캐싱**하고 있어 발생한 오탐으로 진단 — "커널 재시작 후 전체 재실행" 안내.
- 커밋 충돌(mtime 불일치)로 재스테이징하는 과정에서 `plot_training_curve()`에 추가했던
  `title`/`xlabel` 파라미터가 유실될 뻔했으나, 함수 시그니처 재검증으로 자체 발견하여
  복구.

---

## 4단계 (2026-09-16) — 03 노트북 실제 실행 결과 검증

사용자가 "전부 실행했는데 확인해봐"라고 요청 → 03 노트북과 결과 CSV를 디바이스에서
스테이징하여 검증:

- 셀 1~15 오류 없이 순차 실행 확인(`execution_count` 연속).
- **Test 성능 (R² 내림차순)**

  | 순위 | 모델 | Test R² |
  |---|---|---|
  | 1 | **Lasso** | **0.1186** |
  | 2 | Ridge | 0.0368 |
  | 3 | SGDRegressor | 0.0305 |
  | 4 | LinearRegression | 0.0036 |
  | 5 | GradientBoosting | -0.0014 |
  | 6 | DecisionTree | -0.1329 |
  | 7 | RandomForest | -0.3150 |

- **Lasso가 최종 선정 모델**, Test R²=0.1186으로 01 베이스라인(0.0414) 대비 약 2.9배 개선.
- LinearRegression 계수 중 하나가 약 1.6×10¹³으로 비정상적으로 크게 나타남 → 잔존
  다중공선성의 증거로 해석.
- RandomForest(`max_depth` 미제한)와 GradientBoosting(1번째 트리 이후 성능 악화)에서
  심각한 과적합 확인.

---

## 5단계 (2026-09-16) — 1차 보고서 작성 및 배포

- `reports/01_model_comparison_report.md` 작성: 파이프라인 요약, 전체 val/test 비교표,
  핵심 발견(Lasso 1위, LinearRegression 계수 불안정성, 트리모델 과적합, 피처 중요도),
  한계, 다음 단계 제안, 결론 포함.
- 한글 PDF 변환을 위해 `pandoc + xelatex` 경로를 시도했으나 `xeCJK` 등 대용량 LaTeX 패키지
  설치 문제로 **reportlab + fonts-nanum(NanumGothic, TrueType)** 방식으로 전환하여
  `01_model_comparison_report.pdf`(7페이지) 생성. 폰트 패밀리 매핑을 등록하지 않아 `<b>`
  굵게 표시가 적용되지 않는 문제를 시각 검증 중 발견하고 수정.
- `README.md`의 "진행 현황" 섹션을 03 실행 완료 및 1차 보고서 작성 완료로 갱신.
- 모든 산출물(`README.md`, `reports/*.md`, `reports/*.pdf`)을 사용자 컴퓨터의 `pre-kamp2`
  폴더에 커밋 완료. 보고서 마크다운은 Claude 프로젝트 문서로도 저장.

---

## 6단계 (2026-09-16) — 후속 개선 계획 수립 (진행 예정)

1차 보고서의 "다음 단계 제안"을 사용자에게 요약 설명한 뒤, 사용자가 **"전부 다
진행해줘"**라고 요청. 이에 따라 아래 6가지 항목을 구현하기로 결정:

1. RandomForest `max_depth` 튜닝 (현재 미탐색 → 그리드서치에 포함)
2. Lasso가 선택한 피처만으로 다른 모델 재학습
3. K-Fold 교차검증 도입 (안정성 검증용)
4. VIF 기반 다중공선성 재점검
5. XGBoost, LightGBM 등 부스팅 계열 모델 추가
6. Ridge/Lasso/GradientBoosting 피처 중요도 PNG 저장

구현 방식에 대한 확인 질문(AskUserQuestion) 및 사용자 답변:

- **노트북 구성**: 03은 실행 기록이 보고서 근거이므로 그대로 보존하고, **새 노트북
  `04_advanced_modeling.ipynb`**를 만들어 위 6가지를 구현하기로 결정.
- **외부 라이브러리**: XGBoost, LightGBM, statsmodels(VIF 계산용) **전부 설치**하고
  `requirements.txt`/`requirements-windows.txt`에 버전 명시하기로 결정.
- **K-Fold 활용 방식**: 기존 Validation set 기반 하이퍼파라미터 탐색 설계 원칙은 그대로
  유지하고, K-Fold 교차검증은 **최종 선정 모델들의 안정성을 검증하는 용도로 병행**
  실행하기로 결정 (하이퍼파라미터 탐색 방식 자체를 교체하지 않음).

**다음 작업**: `04_advanced_modeling.ipynb` 및 `training_utils.py` 확장 개발, 합성
데이터 드라이런 검증, `requirements.txt` 갱신, `README.md` 갱신 후 사용자 컴퓨터에 배포
예정.